In [1]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
import probeinterface

from probeinterface import Probe, ProbeGroup

import os
import numpy as np
from spikeinterface.core import concatenate_recordings

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import pearsonr
import pandas as pd
import numpy as np
from matplotlib.collections import LineCollection
from probeinterface import write_probeinterface, read_probeinterface
import spikeinterface.exporters as sexp
from spikeinterface.core import write_binary_recording
from pathlib import Path
import pickle
from tabnanny import verbose
import spikeinterface as si
import numpy as np
from spikeinterface.core import get_template_extremum_channel
import scipy.spatial.distance
from scipy.sparse.csgraph import connected_components
import pickle
import pandas as pd
from scipy.io import loadmat

probe_data = loadmat("/media/ubuntu/sda/duan/raw_data/chanMap_DCX_5mm.mat")
probe_x = probe_data['xcoords']
probe_y = probe_data['ycoords']

probe_position = pd.DataFrame(probe_x)
probe_position[1] = probe_y
probe_position['chan_map'] = probe_data['chanMap0ind'].astype(int)

chan_map = pd.read_csv('/media/ubuntu/sda/duan/raw_data/ch_map_R.csv')
merged = chan_map.merge(probe_position, left_on='probeloc', right_on='chan_map')\
                 .iloc[chan_map.index]\
                 .reset_index(drop=True)

probe = Probe()
probe.set_contacts(positions=merged.iloc[:, 2:4])
probe.set_device_channel_indices(range(256))


/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# recording_raw = se.read_intan(f"/home/ubuntu/Documents/jct/project/251205/M190011_260121_150111_merged_130.rhd", stream_id= '0', ignore_integrity_checks=True)

# print('read success')

# recording_raw = spre.unsigned_to_signed(recording_raw)
# recording_raw = spre.resample(recording_raw, 10000)

# recording_recorded = spre.bandpass_filter(recording_raw, freq_min=300, freq_max=3000)
# recording_recorded = spre.notch_filter(recording_recorded, freq=50)
# recording_f = spre.common_reference(recording_recorded, reference="global", operator="median")

# recording_f = recording_f.set_probegroup(probe)

# rec_params_raw = pd.read_csv("/media/ubuntu/sda/duan/result/260121/rec_params.csv")
# rec_params_raw = rec_params_raw[rec_params_raw['bhv_codes'] == 10]

# original_fs = 30000
# target_fs = 10000
# fs_ratio = original_fs / target_fs
# rec_params_raw['rec_codes_points_10000'] = (rec_params_raw['rec_codes_points'] / fs_ratio).astype(int)

# trials_per_group = 1000
# num_groups_to_process = 5
# buffer_seconds = 10
# fs = recording_f.get_sampling_frequency()
# buffer_samples = int(buffer_seconds * fs)

# first_group_min_sample = None
# last_group_max_sample = None

# for group_idx in range(num_groups_to_process):
#     trial_start = group_idx * trials_per_group + 1
#     trial_end = (group_idx + 1) * trials_per_group
#     group_params = rec_params_raw[(rec_params_raw['trial_ids'] >= trial_start) & (rec_params_raw['trial_ids'] <= trial_end)]
#     if len(group_params) == 0:
#         continue

#     min_sample = int(group_params['rec_codes_points_10000'].min())
#     max_sample = int(group_params['rec_codes_points_10000'].max())
#     if group_idx == 0:
#         first_group_min_sample = min_sample
#     if group_idx == num_groups_to_process - 1:
#         last_group_max_sample = max_sample

# if first_group_min_sample is None or last_group_max_sample is None:
#     raise RuntimeError("Could not determine time range for first 5 groups")

# start_sample = max(0, first_group_min_sample - buffer_samples)
# end_sample = min(recording_f.get_num_samples(), last_group_max_sample + buffer_samples)
# print(f"Slicing recording for groups 1-{num_groups_to_process}: {start_sample} to {end_sample} samples")

# recording_segment = recording_f.frame_slice(start_frame=start_sample, end_frame=end_sample)
# recording_preprocessed = recording_segment.save(format="binary", n_jobs=30)

In [3]:
# output_dir = '/home/ubuntu/Documents/jct/project/sorted/260121/groups_01_to_05_sliced/'

# sampling_frequency = recording_preprocessed.get_sampling_frequency()

# phy_folder = f'{output_dir}/phy_folder_for_kilosort'

# print("读取统一的sorting结果...")
# sorting_curated_phy = se.read_phy(phy_folder, exclude_cluster_groups=["noise"])
# print(f"读取到 {len(sorting_curated_phy.unit_ids)} 个units\n")

# print("创建analyzer并计算extensions...")
# analyzer_curated_phy = si.create_sorting_analyzer(
#     sorting=sorting_curated_phy, 
#     recording=recording_preprocessed,  # 使用common reference后的recording
#     format='binary_folder',
#     folder=output_dir + '/analyzer_curated_temp',
#     n_jobs=20, verbose = False
# )

# extensions_to_compute = [
#     "random_spikes",
#     "waveforms",
#     "templates",
#     "unit_locations",
#     "template_similarity"
# ]

# extension_params = {
#     "unit_locations": {"method": "center_of_mass"},
#     "template_similarity": {"method": "cosine_similarity"}
# }

# analyzer_curated_phy.compute(extensions_to_compute, extension_params=extension_params, n_jobs=20, verbose = False)
# print("完成extensions计算\n")

# # 获取neuron信息（整个recording）
# templates_ext = analyzer_curated_phy.get_extension("templates")
# templates_dense = templates_ext.data["average"]
# sparsity = analyzer_curated_phy.sparsity
# unit_locations_ext = analyzer_curated_phy.get_extension("unit_locations")
# unit_locations = unit_locations_ext.get_data()
# channel_locations = analyzer_curated_phy.get_channel_locations()

# # 处理merge逻辑
# if unit_locations.shape[1] >= 2:
#     unit_distances = scipy.spatial.distance.cdist(
#         unit_locations[:, :2], 
#         unit_locations[:, :2], 
#         metric="euclidean"
#     )
# else:
#     unit_distances = scipy.spatial.distance.cdist(
#         unit_locations, 
#         unit_locations, 
#         metric="euclidean"
#     )

# template_similarity_ext = analyzer_curated_phy.get_extension("template_similarity")
# template_similarity = template_similarity_ext.get_data()

# distance_threshold = 10.0
# similarity_threshold = 0.95
# num_units = len(analyzer_curated_phy.unit_ids)
# pair_mask = np.zeros((num_units, num_units), dtype=bool)

# for i in range(num_units):
#     for j in range(i + 1, num_units):
#         if unit_distances[i, j] < distance_threshold and template_similarity[i, j] > similarity_threshold:
#             pair_mask[i, j] = True
#             pair_mask[j, i] = True

# n_components, labels = connected_components(
#     csgraph=pair_mask, 
#     directed=False, 
#     return_labels=True
# )

# merge_unit_groups = []
# unit_ids_list = analyzer_curated_phy.unit_ids
# for component_id in range(n_components):
#     unit_indices = np.where(labels == component_id)[0]
#     if len(unit_indices) > 1:
#         group = [unit_ids_list[i] for i in unit_indices]
#         merge_unit_groups.append(group)

# if len(merge_unit_groups) > 0:
#     print(f"发现 {len(merge_unit_groups)} 组需要merge的units，开始merge...")
#     analyzer_merged = analyzer_curated_phy.merge_units(
#         merge_unit_groups=merge_unit_groups,
#         censor_ms=0.3,
#         merging_mode="hard",
#         new_id_strategy="append",
#         format='binary_folder',
#         folder=output_dir + '/analyzer_merged',
#         verbose=True,
#         n_jobs=20
#     )
    
#     analyzer_merged.compute(extensions_to_compute, extension_params=extension_params, n_jobs=20, verbose = False)
    
#     templates_ext_final = analyzer_merged.get_extension("templates")
#     templates_dense_final = templates_ext_final.data["average"]
#     sparsity_final = analyzer_merged.sparsity
#     unit_locations_ext_final = analyzer_merged.get_extension("unit_locations")
#     unit_locations_final = unit_locations_ext_final.get_data()
#     channel_locations_final = analyzer_merged.get_channel_locations()
#     sorting_final = analyzer_merged.sorting
#     unit_ids_list_final = analyzer_merged.unit_ids
    
#     # 生成position_waveforms
#     position_waveforms_final = []
#     for unit_id in unit_ids_list_final:
#         unit_index = analyzer_merged.sorting.id_to_index(unit_id)
#         template_dense_unit = templates_dense_final[unit_index, :, :]
#         template_sparse_unit = sparsity_final.sparsify_waveforms(template_dense_unit[np.newaxis, :, :], unit_id)[0]
#         sparse_channel_indices = sparsity_final.unit_id_to_channel_indices[unit_id]
        
#         if len(sparse_channel_indices) == 0:
#             position_waveform = np.zeros(templates_dense_final.shape[1], dtype=templates_dense_final.dtype)
#             position_waveforms_final.append(position_waveform)
#             continue
        
#         sparse_channel_locations = channel_locations_final[sparse_channel_indices, :2]
#         unit_location = unit_locations_final[unit_index, :2]
        
#         distances = np.sqrt(np.sum((sparse_channel_locations - unit_location[np.newaxis, :])**2, axis=1))
#         epsilon = 1e-10
#         weights = 1.0 / (distances + epsilon)
#         weights = weights / np.sum(weights)
        
#         position_waveform = np.dot(template_sparse_unit, weights)
#         position_waveforms_final.append(position_waveform)
    
#     position_waveforms_final = np.array(position_waveforms_final)
#     extremum_channels_final = get_template_extremum_channel(
#         analyzer_merged, 
#         peak_sign="neg",
#         outputs="id"
#     )
    
#     channel_ids_list = list(analyzer_merged.recording.get_channel_ids())
# else:
#     print("无需merge units\n")
#     templates_ext_final = analyzer_curated_phy.get_extension("templates")
#     templates_dense_final = templates_ext_final.data["average"]
#     sparsity_final = analyzer_curated_phy.sparsity
#     unit_locations_ext_final = analyzer_curated_phy.get_extension("unit_locations")
#     unit_locations_final = unit_locations_ext_final.get_data()
#     channel_locations_final = analyzer_curated_phy.get_channel_locations()
#     sorting_final = analyzer_curated_phy.sorting
#     unit_ids_list_final = unit_ids_list
    
#     # 生成position_waveforms
#     position_waveforms_final = []
#     for unit_id in unit_ids_list_final:
#         unit_index = analyzer_curated_phy.sorting.id_to_index(unit_id)
#         template_dense_unit = templates_dense_final[unit_index, :, :]
#         template_sparse_unit = sparsity_final.sparsify_waveforms(template_dense_unit[np.newaxis, :, :], unit_id)[0]
#         sparse_channel_indices = sparsity_final.unit_id_to_channel_indices[unit_id]
        
#         if len(sparse_channel_indices) == 0:
#             position_waveform = np.zeros(templates_dense_final.shape[1], dtype=templates_dense_final.dtype)
#             position_waveforms_final.append(position_waveform)
#             continue
        
#         sparse_channel_locations = channel_locations_final[sparse_channel_indices, :2]
#         unit_location = unit_locations_final[unit_index, :2]
        
#         distances = np.sqrt(np.sum((sparse_channel_locations - unit_location[np.newaxis, :])**2, axis=1))
#         epsilon = 1e-10
#         weights = 1.0 / (distances + epsilon)
#         weights = weights / np.sum(weights)
        
#         position_waveform = np.dot(template_sparse_unit, weights)
#         position_waveforms_final.append(position_waveform)
    
#     position_waveforms_final = np.array(position_waveforms_final)
#     extremum_channels_final = get_template_extremum_channel(
#         analyzer_curated_phy, 
#         peak_sign="neg",
#         outputs="id"
#     )
    
#     channel_ids_list = list(analyzer_curated_phy.recording.get_channel_ids())

# print("计算每个unit的channel_id...")
# channel_ids_dict = {} 
# for idx, unit_id in enumerate(unit_ids_list_final):
#     unit_index = sorting_final.id_to_index(unit_id)
#     template_unit = templates_dense_final[unit_index, :, :]  # (n_samples, n_channels)

#     non_zero_channels = []
#     for ch_idx in range(template_unit.shape[1]):  # 遍历channels（最后一个维度）
#         if np.any(template_unit[:, ch_idx] != 0):  # 检查该通道在所有时间点的值
#             # recording的channel_id已经是contact_id，直接使用
#             contact_id = str(channel_ids_list[ch_idx])
#             non_zero_channels.append(contact_id)
    
#     channel_ids_dict[unit_id] = non_zero_channels

# print(f"完成channel_id计算，共处理{len(channel_ids_dict)}个units\n")

# # 计算channel_snr（每个unit的各个channel的SNR）
# print("计算channel_snr...")
# n_channels = recording_preprocessed.get_num_channels()

# # 计算noise_std（使用前10秒的数据）
# duration_samples = int(10 * sampling_frequency)  # 10秒
# max_samples = min(duration_samples, recording_preprocessed.get_num_samples())
# traces = recording_preprocessed.get_traces(start_frame=0, end_frame=max_samples)  # (n_timepoints, n_channels)

# noise_std_detect = np.median(np.abs(traces) / 0.6745, axis=0)  # (n_channels,)

# all_spike_times = []
# all_spike_unit_ids = []
# for unit_id in unit_ids_list_final:
#     spike_train = sorting_final.get_unit_spike_train(unit_id)
#     all_spike_times.extend(spike_train.tolist())
#     all_spike_unit_ids.extend([unit_id] * len(spike_train))

# n_spikes_total = len(all_spike_times)
# n_spikes_sample = min(1000, n_spikes_total)
# if n_spikes_sample > 0:
#     random_indices = np.random.choice(n_spikes_total, size=n_spikes_sample, replace=False)
#     sampled_spike_times = [all_spike_times[i] for i in random_indices]
#     sampled_spike_unit_ids = [all_spike_unit_ids[i] for i in random_indices]
# else:
#     sampled_spike_times = []
#     sampled_spike_unit_ids = []

# left_sample = 10
# right_sample = 20
# window_size = left_sample + right_sample

# channel_snr_dict = {} 

# for unit_id in unit_ids_list_final:
#     channel_snr_dict[unit_id] = {}
#     unit_spike_times = [st for st, uid in zip(sampled_spike_times, sampled_spike_unit_ids) if uid == unit_id]
    
#     if len(unit_spike_times) == 0:
#         unit_spike_times = sorting_final.get_unit_spike_train(unit_id).tolist()
#         if len(unit_spike_times) > 1000:
#             unit_spike_times = np.random.choice(unit_spike_times, size=1000, replace=False).tolist()
    
#     unit_waveforms = []  # List of (n_channels, window_size)
#     valid_spike_times = []
    
#     for spike_time in unit_spike_times:
#         start = spike_time - left_sample
#         end = spike_time + right_sample

#         if start < 0:
#             start = 0
#         if end > recording_preprocessed.get_num_samples():
#             end = recording_preprocessed.get_num_samples()
        
#         waveform = recording_preprocessed.get_traces(start_frame=start, end_frame=end)  # (n_timepoints, n_channels)
#         unit_waveforms.append(waveform)
#         valid_spike_times.append(spike_time)
    
#     if len(unit_waveforms) == 0:
#         continue
    
#     unit_waveforms = np.array(unit_waveforms)  # (n_spikes, n_timepoints, n_channels)
    
#     spike_time_values = unit_waveforms[:, left_sample, :]  # (n_spikes, n_channels) - 每个spike在spike_time时刻各个channel的值
    
#     channel_amplitudes = np.mean(spike_time_values, axis=0)  # (n_channels,) - 每个channel的平均值（在spike_time时刻）
#     channel_snr = np.abs(channel_amplitudes) / noise_std_detect  # (n_channels,)
    
#     unit_channel_ids = channel_ids_dict.get(unit_id, []) 
    
#     for ch_idx, snr_value in enumerate(channel_snr):
#         channel_id = str(channel_ids_list[ch_idx])
#         # 只保存 channel_ids_dict 中列出的通道
#         if channel_id in unit_channel_ids:
#             channel_snr_dict[unit_id][channel_id] = float(snr_value)

# print(f"完成channel_snr计算，共处理{len(channel_snr_dict)}个units\n")

# neuron_inf_all = {}
# for idx, unit_id in enumerate(unit_ids_list_final):
#     neuron_inf_all[unit_id] = {
#         'location_x': float(unit_locations_final[idx, 0]),
#         'location_y': float(unit_locations_final[idx, 1]),
#         'position_waveform': position_waveforms_final[idx],
#         'extremum_channel': extremum_channels_final[unit_id],
#         'channel_id': channel_ids_dict[unit_id],
#         'channel_snr': channel_snr_dict.get(unit_id, {})  # 添加channel_snr字段
#     }

# print("生成整体的gt_detect_array...")
# spike_vector_final = sorting_final.to_spike_vector()
# gt_detect_data_all = []

# for spike in spike_vector_final:
#     unit_index = spike['unit_index']
#     unit_id = sorting_final.unit_ids[unit_index]
#     sample_index = spike['sample_index']  
    
    
#     extremum_channel = extremum_channels_final[unit_id]
    
#     gt_detect_data_all.append({
#         'time': sample_index,
#         'unit_id': unit_id,
#         'extremum_channel': str(extremum_channel),
#     })

# gt_detect_array_all = pd.DataFrame(gt_detect_data_all)
# print(f"完成gt_detect_array生成，共{len(gt_detect_array_all)}个spikes\n")

# print("保存整体的neuron_inf_all和gt_detect_array_all...")
# with open(output_dir + '/neuron_inf_all.pickle', 'wb') as f:
#     pickle.dump(neuron_inf_all, f)
# gt_detect_array_all.to_csv(output_dir + '/gt_detect_array_all.csv', index=False)


In [4]:
output_dir = '/home/ubuntu/Documents/jct/project/sorted/260121/groups_01_to_05_sliced/history'

with open(output_dir + '/neuron_inf.pickle', 'rb') as f:
    neuron_inf = pickle.load(f)

gt_detect_array = pd.read_csv(output_dir + '/gt_detect_array.csv')

In [ ]:
with PdfPages(f"{output_dir}/unit_histplots.pdf") as pdf:
    for unit in gt_detect_array['unit_id'].unique():
        sample_max = gt_detect_array['sample_index'].max()
        temp = gt_detect_array[gt_detect_array['unit_id'] == unit]
        plt.figure(figsize=(6, 4))
        sns.histplot(temp['sample_index'])
        plt.xlim(0, sample_max)
        plt.title(unit)
        plt.xlabel('Sample Index')
        pdf.savefig()
        plt.close()



In [5]:
rec_params = pd.read_csv("/media/ubuntu/sda/duan/result/260121/rec_params.csv")
condition = pd.read_csv("/media/ubuntu/sda/duan/result/260121/images_sequence_10000.csv")

rec_params = rec_params[rec_params['bhv_codes'] == 10]
rec_params.index = range(10000)
rec_params = pd.concat((rec_params, condition), axis = 1)
# rec_params['trial_condition'] = rec_params['trial_condition'].astype(int)

rec_params['rec_codes_points'] = rec_params['rec_codes_points'] / 3 
rec_params['rec_codes_points'] = rec_params['rec_codes_points'].astype(int)


rec_params = rec_params[rec_params['trial_error'] == 0.0]


In [6]:
from joblib import Parallel, delayed
from tqdm import tqdm

output_dir = '/home/ubuntu/Documents/jct/project/sorted/260121/groups_01_to_05_sliced/history'

# Time windows in samples (at 10kHz sampling rate)
window_before_samples = 1500
window_after_samples = 4500
sample_to_ms = 0.1  # 10kHz = 0.1ms per sample
SAMPLING_RATE = 10000

# Convert to milliseconds
window_before_ms = int(window_before_samples * sample_to_ms)  # 150 ms
window_after_ms = int(window_after_samples * sample_to_ms)    # 450 ms
total_time_ms = window_before_ms + window_after_ms            # 600 ms

# Raster matrix: bin by milliseconds (1ms per bin)
n_time_bins_ms = total_time_ms  # 600 bins, each representing 1ms

# PSTH smoothing window
psth_window_size_ms = 20  # 20ms smoothing window

all_neuron_ids = sorted(gt_detect_array['unit_id'].unique())
trial_stim_points = rec_params['rec_codes_points'].astype(int).values
stimulus_ids = rec_params['image_name'].astype(str).values

stimulus_unique = pd.unique(stimulus_ids)
stimulus_id_to_index = {sid: idx for idx, sid in enumerate(stimulus_unique)}

n_trials = len(trial_stim_points)
n_neurons = len(all_neuron_ids)

# Helper function for raster matrix computation per neuron
def compute_raster_for_neuron(neuron_id, neuron_idx, gt_detect_array, trial_stim_points, 
                               window_before_samples, window_after_samples, 
                               window_before_ms, n_time_bins_ms, sample_to_ms):
    """Compute raster matrix for a single neuron (binned by milliseconds)"""
    neuron_spikes = gt_detect_array.loc[gt_detect_array['unit_id'] == neuron_id, 'sample_index'].values.astype(np.int64)
    neuron_spikes.sort()
    
    neuron_raster = np.zeros((n_trials, n_time_bins_ms), dtype=np.float32)
    
    for trial_idx, stim_point in enumerate(trial_stim_points):
        start_ext_samples = stim_point - window_before_samples
        end_ext_samples = stim_point + window_after_samples
        
        # Get spikes in the time window
        trial_spikes_samples = neuron_spikes[(neuron_spikes >= start_ext_samples) & (neuron_spikes <= end_ext_samples)]
        
        # Convert spike times to milliseconds relative to window start
        trial_spikes_ms = (trial_spikes_samples - start_ext_samples) * sample_to_ms
        
        # Bin spikes into 1ms bins
        for spike_ms in trial_spikes_ms:
            bin_idx = int(np.floor(spike_ms))
            if 0 <= bin_idx < n_time_bins_ms:
                neuron_raster[trial_idx, bin_idx] += 1
    
    return neuron_idx, neuron_raster

# Parallel computation of raster matrix
print("Computing raster matrix (parallelized, binned by milliseconds)...")
raster_results = Parallel(n_jobs=-1, backend='threading')(
    delayed(compute_raster_for_neuron)(
        neuron_id, neuron_idx, gt_detect_array, trial_stim_points,
        window_before_samples, window_after_samples,
        window_before_ms, n_time_bins_ms, sample_to_ms
    )
    for neuron_idx, neuron_id in enumerate(tqdm(all_neuron_ids, desc="Raster computation"))
)

all_trial_raster_matrix = np.zeros((n_trials, n_neurons, n_time_bins_ms), dtype=np.float32)
for neuron_idx, neuron_raster in raster_results:
    all_trial_raster_matrix[:, neuron_idx, :] = neuron_raster

# Helper function for PSTH matrix computation per neuron
def compute_psth_for_neuron(neuron_idx, all_trial_raster_matrix, psth_window_size_ms, n_time_bins_ms):
    """
    Compute PSTH matrix for a single neuron (using ms-based raster)
    
    Uses a sliding window approach: for each 1ms bin, calculate firing rate using
    a psth_window_size_ms (e.g., 20ms) window centered on that bin.
    
    This matches the MATLAB reference implementation:
    - Each bin represents 1ms
    - For each time point, use a sliding window of psth_window_size_ms
    - Window is centered on the time point (e.g., for 20ms window at time 50ms: 
      uses bins 40-60, which is 21 bins total, representing ~20ms centered at 50ms)
    """
    raster_raw = all_trial_raster_matrix[:, neuron_idx, :]
    neuron_psth = np.zeros((n_trials, n_time_bins_ms), dtype=np.float32)
    
    # Convert to 0-based indexing for Python (MATLAB uses 1-based)
    for time_point_ms in range(n_time_bins_ms):
        # MATLAB equivalent: time_points-psth_window_size_ms/2:time_points+psth_window_size_ms/2
        # For 20ms window at time_point_ms=50: window = 40:60 (21 bins, centered at 50)
        time_point_1based = time_point_ms + 1  # Convert to 1-based for comparison
        
        # Calculate window boundaries (matching MATLAB logic)
        if time_point_1based - psth_window_size_ms // 2 < 1:
            # Near start: use first psth_window_size_ms bins
            window_start_ms = 0
            window_end_ms = psth_window_size_ms
        elif time_point_1based + psth_window_size_ms // 2 > n_time_bins_ms:
            # Near end: use last psth_window_size_ms bins
            window_start_ms = n_time_bins_ms - psth_window_size_ms
            window_end_ms = n_time_bins_ms
        else:
            # Middle: centered window
            window_start_ms = time_point_ms - psth_window_size_ms // 2
            window_end_ms = time_point_ms + psth_window_size_ms // 2 + 1
        
        time_window = np.arange(window_start_ms, window_end_ms)
        
        # PSTH: sum spikes in window and convert to Hz (spikes per second)
        # Since raster is already in spikes per 1ms bin, we multiply by 1000 to get Hz
        # Formula: (spikes in window / window_size_ms) * 1000 = Hz
        neuron_psth[:, time_point_ms] = (
            1000 * np.sum(raster_raw[:, time_window], axis=1) / len(time_window)
        )
    
    return neuron_idx, neuron_psth

# Parallel computation of PSTH matrix
print("Computing PSTH matrix (parallelized)...")
psth_results = Parallel(n_jobs=-1, backend='threading')(
    delayed(compute_psth_for_neuron)(
        neuron_idx, all_trial_raster_matrix, psth_window_size_ms, n_time_bins_ms
    )
    for neuron_idx in tqdm(range(n_neurons), desc="PSTH computation")
)

all_trial_psth_matrix = np.zeros((n_trials, n_neurons, n_time_bins_ms), dtype=np.float32)
for neuron_idx, neuron_psth in psth_results:
    all_trial_psth_matrix[:, neuron_idx, :] = neuron_psth

print(f"Computation completed!")
print(f"Raster matrix shape: {all_trial_raster_matrix.shape} (n_trials, n_neurons, n_time_bins_ms)")
print(f"PSTH matrix shape: {all_trial_psth_matrix.shape} (n_trials, n_neurons, n_time_bins_ms)")
print(f"Time range: -{window_before_ms}ms to +{window_after_ms}ms ({total_time_ms}ms total, {n_time_bins_ms} bins at 1ms/bin)")


Computing raster matrix (parallelized, binned by milliseconds)...


Raster computation: 100%|██████████| 164/164 [00:24<00:00,  6.60it/s]


Computing PSTH matrix (parallelized)...


PSTH computation: 100%|██████████| 164/164 [00:10<00:00, 15.36it/s]


Computation completed!
Raster matrix shape: (8489, 164, 600) (n_trials, n_neurons, n_time_bins_ms)
PSTH matrix shape: (8489, 164, 600) (n_trials, n_neurons, n_time_bins_ms)
Time range: -150ms to +450ms (600ms total, 600 bins at 1ms/bin)


In [7]:
all_trial_psth_matrix_dict = {}
trial_image_dict = {}

print("按neuron筛选trials（总firing rate >= 1）...")

for neuron_idx, neuron_id in enumerate(all_neuron_ids):
    neuron_psth = all_trial_psth_matrix[:, neuron_idx, :]
    
    total_firing_rates = np.sum(neuron_psth, axis=1)
    
    valid_trial_mask = total_firing_rates >= 20.0
    valid_trial_indices = np.where(valid_trial_mask)[0]
    
    filtered_psth = neuron_psth[valid_trial_indices, :]
    filtered_images = stimulus_ids[valid_trial_indices]

    if filtered_psth.shape[0] >= 1000:
        all_trial_psth_matrix_dict[neuron_id] = filtered_psth
        trial_image_dict[neuron_id] = filtered_images
    

print(f"\n筛选完成！")
print(f"  PSTH形状: {all_trial_psth_matrix_dict[all_neuron_ids[1]].shape}")


按neuron筛选trials（总firing rate >= 1）...

筛选完成！
  PSTH形状: (4444, 600)


In [8]:
from scipy.stats import ranksums

baseline_start_ms = -30
baseline_end_ms = 30
response_window1_start_ms = 50
response_window1_end_ms = 120
response_window2_start_ms = 120
response_window2_end_ms = 240

# Re-define window_before_ms (raster matrix is now binned by milliseconds)
window_before_ms = 150  # milliseconds before stimulus

# Convert time windows to raster matrix indices (raster is binned by milliseconds: 1ms per bin)
# Index = relative_time_ms + window_before_ms
baseline_start_idx = int(baseline_start_ms + window_before_ms)
baseline_end_idx = int(baseline_end_ms + window_before_ms) + 1
response1_start_idx = int(response_window1_start_ms + window_before_ms)
response1_end_idx = int(response_window1_end_ms + window_before_ms) + 1
response2_start_idx = int(response_window2_start_ms + window_before_ms)
response2_end_idx = int(response_window2_end_ms + window_before_ms) + 1

if baseline_start_idx >= baseline_end_idx:
    baseline_start_idx, baseline_end_idx = baseline_end_idx, baseline_start_idx + 1

print(f"时间窗口索引 (相对于 raster 矩阵，单位: ms bins):")
print(f"  Baseline: {baseline_start_idx}-{baseline_end_idx} ({baseline_start_ms} to {baseline_end_ms} ms)")
print(f"  Response window 1: {response1_start_idx}-{response1_end_idx} ({response_window1_start_ms} to {response_window1_end_ms} ms)")
print(f"  Response window 2: {response2_start_idx}-{response2_end_idx} ({response_window2_start_ms} to {response_window2_end_ms} ms)")

all_trial_psth_matrix_dict_filtered = {}
trial_image_dict_filtered = {}

print(f"\n进行Wilcoxon rank-sum test识别响应神经元...")
print(f"筛选前neuron数量: {len(all_trial_psth_matrix_dict)}")

for neuron_idx, neuron_id in enumerate(all_neuron_ids):
    if neuron_id not in all_trial_psth_matrix_dict:
        continue
    
    neuron_psth = all_trial_psth_matrix_dict[neuron_id]
    neuron_raster = all_trial_raster_matrix[:, neuron_idx, :]
    
    neuron_images = trial_image_dict[neuron_id]
    valid_trial_indices = np.where(np.isin(stimulus_ids, neuron_images))[0]
    
    if len(valid_trial_indices) == 0:
        continue
    
    neuron_raster_filtered = neuron_raster[valid_trial_indices, :]
    
    baseline_firing_rates = np.mean(neuron_raster_filtered[:, baseline_start_idx:baseline_end_idx], axis=1)
    response1_firing_rates = np.mean(neuron_raster_filtered[:, response1_start_idx:response1_end_idx], axis=1)
    response2_firing_rates = np.mean(neuron_raster_filtered[:, response2_start_idx:response2_end_idx], axis=1)
    
    stat1, p_val1 = ranksums(baseline_firing_rates, response1_firing_rates, alternative='two-sided')
    stat2, p_val2 = ranksums(baseline_firing_rates, response2_firing_rates, alternative='two-sided')
    
    min_p_val = min(p_val1, p_val2)
    
    if min_p_val < 0.001:
        all_trial_psth_matrix_dict_filtered[neuron_id] = neuron_psth
        trial_image_dict_filtered[neuron_id] = neuron_images

print(f"筛选后neuron数量: {len(all_trial_psth_matrix_dict_filtered)}")
print(f"保留的neuron比例: {len(all_trial_psth_matrix_dict_filtered)/len(all_trial_psth_matrix_dict)*100:.1f}%")

all_trial_psth_matrix_dict = all_trial_psth_matrix_dict_filtered
trial_image_dict = trial_image_dict_filtered


时间窗口索引 (相对于 raster 矩阵，单位: ms bins):
  Baseline: 120-181 (-30 to 30 ms)
  Response window 1: 200-271 (50 to 120 ms)
  Response window 2: 270-391 (120 to 240 ms)

进行Wilcoxon rank-sum test识别响应神经元...
筛选前neuron数量: 164
筛选后neuron数量: 131
保留的neuron比例: 79.9%


In [9]:
filtered_neuron_ids = sorted(all_trial_psth_matrix_dict.keys())
n_filtered_neurons = len(filtered_neuron_ids)
n_images = len(stimulus_unique)
n_time_bins = all_trial_psth_matrix_dict[filtered_neuron_ids[0]].shape[1]

neuron_image_response_matrix = np.zeros((n_filtered_neurons, n_images, n_time_bins), dtype=np.float32)
neuron_image_baseline = np.zeros((n_filtered_neurons, n_images), dtype=np.float32)
image_to_index = {img: idx for idx, img in enumerate(stimulus_unique)}

for neuron_idx, neuron_id in enumerate(filtered_neuron_ids):
    neuron_psth = all_trial_psth_matrix_dict[neuron_id]
    neuron_images = trial_image_dict[neuron_id]
    
    for image_name in stimulus_unique:
        image_idx = image_to_index[image_name]
        image_mask = neuron_images == image_name
        image_trials = neuron_psth[image_mask, :]
        
        if len(image_trials) > 0:
            neuron_image_response_matrix[neuron_idx, image_idx, :] = np.mean(image_trials, axis=0)
            # 计算baseline平均值（所有trial的baseline响应平均值）
            neuron_image_baseline[neuron_idx, image_idx] = np.mean(
                np.mean(image_trials[:, baseline_start_idx:baseline_end_idx], axis=1)
            )

# 计算减去baseline的PSTH
neuron_image_response_matrix_baseline_subtracted = neuron_image_response_matrix - neuron_image_baseline[:, :, np.newaxis]

from scipy.ndimage import gaussian_filter1d
n_bins_target = 30
gaussian_sigma_bins = 10

# 对原始PSTH进行高斯平滑和聚合
neuron_image_response_matrix = gaussian_filter1d(
    neuron_image_response_matrix, sigma=gaussian_sigma_bins, axis=2, mode="nearest"
)
bin_size = n_time_bins // n_bins_target
n_valid = n_bins_target * bin_size
neuron_image_response_matrix = (
    neuron_image_response_matrix[:, :, :n_valid]
    .reshape(n_filtered_neurons, n_images, n_bins_target, bin_size)
    .mean(axis=-1)
)

# 对减去baseline的PSTH进行高斯平滑和聚合
neuron_image_response_matrix_baseline_subtracted = gaussian_filter1d(
    neuron_image_response_matrix_baseline_subtracted, sigma=gaussian_sigma_bins, axis=2, mode="nearest"
)
neuron_image_response_matrix_baseline_subtracted = (
    neuron_image_response_matrix_baseline_subtracted[:, :, :n_valid]
    .reshape(n_filtered_neurons, n_images, n_bins_target, bin_size)
    .mean(axis=-1)
)

print(f"\n响应矩阵形状: {neuron_image_response_matrix.shape} (时间维已高斯平滑并聚合为{n_bins_target}个bin)")
print(f"减去baseline的响应矩阵形状: {neuron_image_response_matrix_baseline_subtracted.shape}")



响应矩阵形状: (131, 1000, 30) (时间维已高斯平滑并聚合为30个bin)
减去baseline的响应矩阵形状: (131, 1000, 30)


In [10]:

baseline_start_ms = -30
baseline_end_ms = 30
response_window1_start_ms = 50
response_window1_end_ms = 120
response_window2_start_ms = 120
response_window2_end_ms = 240

# 原始PSTH矩阵信息
window_before_ms = 150  # 刺激前的时间
total_time_ms = 600     # 总时间长度
n_time_bins_original = 600  # 原始时间bin数（1ms per bin）
n_time_bins_aggregated = 30  # 聚合后的时间bin数

# 计算每个聚合bin对应的原始时间范围
bin_duration_ms = total_time_ms / n_time_bins_aggregated  # 20ms per bin

def ms_to_aggregated_bin(ms_value):
    """将毫秒值转换为聚合后的bin索引（0-based）"""
    # 时间轴从 -150ms 到 +450ms
    # 相对时间 = ms_value + window_before_ms
    relative_time = ms_value + window_before_ms
    # 聚合bin索引 = 相对时间 / bin_duration_ms
    bin_idx = int(relative_time / bin_duration_ms)
    # 确保索引在有效范围内
    return max(0, min(n_time_bins_aggregated - 1, bin_idx))

baseline_start_bin = ms_to_aggregated_bin(baseline_start_ms)
baseline_end_bin = ms_to_aggregated_bin(baseline_end_ms)
response1_start_bin = ms_to_aggregated_bin(response_window1_start_ms)
response1_end_bin = ms_to_aggregated_bin(response_window1_end_ms)
response2_start_bin = ms_to_aggregated_bin(response_window2_start_ms)
response2_end_bin = ms_to_aggregated_bin(response_window2_end_ms)

print(f"\n聚合bin的时间范围验证:")
for bin_idx in range(n_time_bins_aggregated):
    bin_start_ms = bin_idx * bin_duration_ms - window_before_ms
    bin_end_ms = (bin_idx + 1) * bin_duration_ms - window_before_ms
    if bin_idx in [baseline_start_bin, response1_start_bin, response2_start_bin]:
        print(f"  Bin {bin_idx}: {bin_start_ms:.1f} to {bin_end_ms:.1f} ms ← 窗口起始点")

n_neurons = neuron_image_response_matrix.shape[0]  # 131
n_images = neuron_image_response_matrix.shape[1]   # 1000

baseline_responses = np.zeros((n_neurons,), dtype=np.float32)
response1_responses = np.zeros((n_neurons, n_images), dtype=np.float32)
response2_responses = np.zeros((n_neurons, n_images), dtype=np.float32)

for neuron_idx, neuron_id in enumerate(filtered_neuron_ids):
    neuron_psth = all_trial_psth_matrix_dict[neuron_id]
    neuron_images = trial_image_dict[neuron_id]

    baseline_responses[neuron_idx] = np.mean(
        np.mean(neuron_psth[:, baseline_start_idx:baseline_end_idx], axis=1)
    )
    
    for image_idx, image_name in enumerate(stimulus_unique):
        image_mask = neuron_images == image_name
        image_trials_psth = neuron_psth[image_mask, :]
        
        if len(image_trials_psth) > 0:
            response1_responses[neuron_idx, image_idx] = np.mean(
                np.mean(image_trials_psth[:, response1_start_idx:response1_end_idx], axis=1)
            )
            response2_responses[neuron_idx, image_idx] = np.mean(
                np.mean(image_trials_psth[:, response2_start_idx:response2_end_idx], axis=1)
            )


print("\n计算相对于baseline的倍数...")

epsilon = 1e-10
baseline_safe = np.maximum(baseline_responses, epsilon)[:, None]

response1_to_baseline_ratio = response1_responses / baseline_safe
response2_to_baseline_ratio = response2_responses / baseline_safe

response_ratios = {
    'baseline_responses': baseline_responses,
    'response1_responses': response1_responses,
    'response2_responses': response2_responses,
    'response1_to_baseline_ratio': response1_to_baseline_ratio,
    'response2_to_baseline_ratio': response2_to_baseline_ratio,
}

output_file = f"{output_dir}/response_ratios.pkl"
with open(output_file, 'wb') as f:
    pickle.dump(response_ratios, f)



聚合bin的时间范围验证:
  Bin 6: -30.0 to -10.0 ms ← 窗口起始点
  Bin 10: 50.0 to 70.0 ms ← 窗口起始点
  Bin 13: 110.0 to 130.0 ms ← 窗口起始点

计算相对于baseline的倍数...


In [13]:
import torch
import math
import numpy as np

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from PIL import Image

class TemporalEPEncoder(nn.Module):
    def __init__(self, input_dim=244, time_bins=30, d_model=256, n_token=128, 
                 num_conv_layers=3, dropout=0.2, output_dim=768):
        super().__init__()
        self.input_dim = input_dim
        self.time_bins = time_bins
        self.d_model = d_model
        self.n_token = n_token
        self.output_dim = output_dim
        
        if input_dim > 200:
            hidden_dim = min(input_dim // 4, d_model * 8)
            self.input_proj = nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.LayerNorm(hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, d_model * 4),
                nn.LayerNorm(d_model * 4),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(d_model * 4, d_model),
            )
        else:
            self.input_proj = nn.Sequential(
                nn.Linear(input_dim, d_model * 4),
                nn.LayerNorm(d_model * 4),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(d_model * 4, d_model),
            )
        
        conv_layers = []
        for i in range(num_conv_layers):
            if i == 0:
                in_channels = d_model
            else:
                in_channels = d_model * 2
            
            if i == num_conv_layers - 1:
                out_channels = d_model
            else:
                out_channels = d_model * 2
            
            conv_layers.extend([
                nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm1d(out_channels),
                nn.GELU(),
                nn.Dropout(dropout)
            ])
        self.temporal_conv = nn.Sequential(*conv_layers)
        
        self.adaptive_pool = nn.AdaptiveAvgPool1d(n_token)
        
        self.final_proj = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout * 0.5)
        )
        
        self.feature_proj = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.LayerNorm(d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, output_dim),
            nn.LayerNorm(output_dim)
        )
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.5)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, (nn.BatchNorm1d, nn.LayerNorm)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        if torch.isnan(x).any() or torch.isinf(x).any():
            x = torch.nan_to_num(x, nan=0.0, posinf=1.0, neginf=-1.0)
        
        x = self.input_proj(x)
        x = x.transpose(1, 2)
        x = self.temporal_conv(x)
        x = self.adaptive_pool(x)
        x = x.transpose(1, 2)
        x = self.final_proj(x)
        
        x = self.feature_proj(x)
        feature = x.mean(dim=1)
        
        return feature


class PSTHClusterClassifier(nn.Module):
    def __init__(self, encoder, num_classes=7):
        super().__init__()
        self.encoder = encoder
        feat_dim = getattr(encoder, 'output_dim', 768)
        self.classifier = nn.Linear(feat_dim, num_classes)
    def forward(self, x, return_features=False):
        feat = self.encoder(x)
        logits = self.classifier(feat)
        if return_features:
            return logits, feat
        return logits



In [12]:
from typing import Any


class ContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature
        
    def forward(self, features_1, features_2):
        features_1 = features_1.float()
        features_2 = features_2.float()
        
        features_1 = F.normalize(features_1, dim=1)
        features_2 = F.normalize(features_2, dim=1)
        
        logits_12 = torch.matmul(features_1, features_2.T) / self.temperature
        logits_21 = torch.matmul(features_2, features_1.T) / self.temperature
        
        batch_size = features_1.size(0)
        labels = torch.arange(batch_size, device=features_1.device)
        
        loss_12 = F.cross_entropy(logits_12, labels)
        loss_21 = F.cross_entropy(logits_21, labels)
        
        return (loss_12 + loss_21) / 2


class InfoNCELoss(nn.Module):
    def __init__(self, temperature=0.07, num_classes=7):
        super().__init__()
        self.temperature = temperature
        self.num_classes = num_classes
        
    def forward(self, features, labels):
        """
        InfoNCE损失用于7类分类
        Args:
            features: (batch_size, feature_dim) 特征向量
            labels: (batch_size,) 类别标签 (0-6 for 7 classes)
        Returns:
            loss: InfoNCE损失值
        """
        batch_size = features.size(0)
        device = features.device
        
        # 归一化特征
        features = F.normalize(features, dim=1)
        
        # 计算所有样本之间的相似度矩阵
        similarity_matrix = torch.matmul(features, features.T) / self.temperature
        
        # 创建正样本掩码：同一类的样本是正样本
        labels_expanded = labels.unsqueeze(1)
        positive_mask = (labels_expanded == labels_expanded.T).float()
        
        # 排除自己（对角线）
        positive_mask.fill_diagonal_(0)
        
        # 计算每个样本的正样本数量
        num_positives = positive_mask.sum(dim=1, keepdim=True)
        
        # 避免除以零，如果没有正样本则设为1
        num_positives = torch.clamp(num_positives, min=1)
        
        # 计算exp(similarity)
        exp_sim = torch.exp(similarity_matrix)
        
        # 对于每个样本，计算正样本的exp和
        pos_exp_sum = (exp_sim * positive_mask).sum(dim=1, keepdim=True)
        
        # 计算所有样本的exp和（包括自己）
        all_exp_sum = exp_sim.sum(dim=1, keepdim=True)
        
        # InfoNCE损失：-log(正样本exp和 / 所有样本exp和)
        # 平均多个正样本
        loss_per_sample = -torch.log((pos_exp_sum / num_positives) / (all_exp_sum + 1e-8) + 1e-8)
        
        return loss_per_sample.mean()


class PSTHDataset(Dataset):
    def __init__(self, psth_data, image_paths, image_names, clip_model, clip_preprocess, device='cpu'):
        self.psth_data = torch.tensor(psth_data, dtype=torch.float32)
        self.image_paths = image_paths
        self.image_names = image_names
        self.clip_model = clip_model
        self.clip_preprocess = clip_preprocess
        self.device = device
        
        self.clip_features_cache = {}
        self._precompute_clip_features()
    
    def _precompute_clip_features(self):
        print("预计算CLIP特征...")
        self.clip_model.eval()
        with torch.no_grad():
            for idx, img_path in enumerate[Any](self.image_paths):
                if idx % 100 == 0:
                    print(f"处理进度: {idx}/{len(self.image_paths)}")
                try:
                    img = Image.open(img_path).convert('RGB')
                    img_tensor = self.clip_preprocess(img).unsqueeze(0).to(self.device)
                    clip_dtype = next(self.clip_model.parameters()).dtype
                    img_tensor = img_tensor.to(clip_dtype)
                    clip_feature = self.clip_model.encode_image(img_tensor)
                    clip_feature = F.normalize(clip_feature, dim=-1)
                    self.clip_features_cache[idx] = clip_feature.cpu()
                except Exception as e:
                    print(f"Error loading image {img_path}: {e}")
                    self.clip_features_cache[idx] = torch.zeros(1, 768)
        print("CLIP特征预计算完成")
    
    def __len__(self):
        return len(self.psth_data)
    
    def __getitem__(self, idx):
        psth = self.psth_data[idx]
        psth = psth.transpose(0, 1)
        clip_feature = self.clip_features_cache[idx].squeeze(0)
        return psth, clip_feature


class PSTHClusterDataset(Dataset):
    def __init__(self, psth_data, cluster_labels):
        self.psth_data = torch.tensor(psth_data, dtype=torch.float32)
        self.cluster_labels = np.array(cluster_labels, dtype=np.int64)
    def __len__(self):
        return len(self.psth_data)
    def __getitem__(self, idx):
        psth = self.psth_data[idx].transpose(0, 1)
        return psth, self.cluster_labels[idx]


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"使用设备: {device}")

PKL_PATH = "/media/ubuntu/sda/Monkey/semantic/epoch_10_monkeyN/merged_cluster_class_counts.pkl"
with open(PKL_PATH, 'rb') as f:
    cluster_data = pickle.load(f)
class_to_cluster = {}
for cluster_id in range(7):
    for class_name in cluster_data[cluster_id]:
        class_to_cluster[class_name] = cluster_id

condition_df = pd.read_csv("/media/ubuntu/sda/duan/result/260121/images_sequence_10000.csv")
image_path_dict = {row['image_name']: row['image_path'] for _, row in condition_df.iterrows()}

valid_psth = []
valid_labels = []

for stim_idx, img_name in enumerate(stimulus_unique):
    if img_name not in image_path_dict:
        continue
    path = image_path_dict[img_name]
    class_name = os.path.basename(os.path.dirname(path))
    if class_name not in class_to_cluster:
        continue
    psth_per_image = neuron_image_response_matrix[:, stim_idx, 4:]
    if psth_per_image.sum() <= 0:
        continue
    valid_psth.append(psth_per_image)
    valid_labels.append(class_to_cluster[class_name])

valid_psth = np.array(valid_psth)
valid_labels = np.array(valid_labels)
print(f"有效数据(7类cluster): {len(valid_psth)} 张图片")
print(f"PSTH形状: {valid_psth.shape}, 各类数量: {np.bincount(valid_labels, minlength=7)}")

train_indices, val_indices = train_test_split(
    range(len(valid_psth)),
    test_size=0.1,
    random_state=42,
    stratify=valid_labels
)

train_psth = valid_psth[train_indices]
train_labels = valid_labels[train_indices]
val_psth = valid_psth[val_indices]
val_labels = valid_labels[val_indices]

print(f"训练集: {len(train_psth)}, 验证集: {len(val_psth)}")

train_dataset = PSTHClusterDataset(train_psth, train_labels)
val_dataset = PSTHClusterDataset(val_psth, val_labels)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)



使用设备: cuda
有效数据(7类cluster): 1000 张图片
PSTH形状: (1000, 131, 26), 各类数量: [ 28 176  32  95 195 217 257]
训练集: 900, 验证集: 100


In [19]:
n_neurons = train_psth.shape[1]
n_time_bins = train_psth.shape[2]

num_runs = 15
num_epochs = 10
top_k = 5
all_runs_results = []

print(f"开始重复训练，共 {num_runs} 次，每次 {num_epochs} 个epoch\n")

for run_idx in range(num_runs):
    print(f"=" * 60)
    print(f"训练第 {run_idx + 1}/{num_runs} 次")
    print(f"=" * 60)
    
    torch.manual_seed(42 + run_idx)
    np.random.seed(42 + run_idx)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(42 + run_idx)
    
    encoder_backbone = TemporalEPEncoder(
        input_dim=n_neurons,
        time_bins=n_time_bins,
        d_model=64,
        n_token=128,
        num_conv_layers=2,
        dropout=0.2,
        output_dim=768
    )
    model = PSTHClusterClassifier(encoder_backbone, num_classes=7).to(device)
    ce_criterion = nn.CrossEntropyLoss()
    infonce_criterion = InfoNCELoss(temperature=0.07, num_classes=7)
    alpha = 0.5
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    
    best_val_acc = 0.0
    best_epoch = 0
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for psth_batch, label_batch in train_loader:
            psth_batch = psth_batch.to(device)
            label_batch = label_batch.to(device)

            optimizer.zero_grad()
            logits, features = model(psth_batch, return_features=True)
            ce_loss = ce_criterion(logits, label_batch)
            infonce_loss = infonce_criterion(features, label_batch)
            loss = alpha * ce_loss + (1 - alpha) * infonce_loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item() * psth_batch.size(0)
            pred = logits.argmax(dim=1)
            train_correct += (pred == label_batch).sum().item()
            train_total += label_batch.size(0)

        avg_train_loss = train_loss / train_total
        train_acc = train_correct / train_total
        scheduler.step()

        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for psth_batch, label_batch in val_loader:
                psth_batch = psth_batch.to(device)
                label_batch = label_batch.to(device)
                logits, features = model(psth_batch, return_features=True)
                ce_loss = ce_criterion(logits, label_batch)
                infonce_loss = infonce_criterion(features, label_batch)
                loss = alpha * ce_loss + (1 - alpha) * infonce_loss
                val_loss += loss.item() * psth_batch.size(0)
                pred = logits.argmax(dim=1)
                val_correct += (pred == label_batch).sum().item()
                val_total += label_batch.size(0)

        avg_val_loss = val_loss / val_total
        val_acc = val_correct / val_total

        # if (epoch + 1) % 1 == 0:
        #     print(f"  Epoch {epoch+1}/{num_epochs} - Train Loss: {avg_train_loss:.4f} Acc: {train_acc:.4f}, Val Loss: {avg_val_loss:.4f} Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch + 1
            torch.save(model.state_dict(), f'{output_dir}/psth_cluster_classifier_run_{run_idx+1}_best.pth')
    
    all_runs_results.append({
        'run_index': run_idx + 1,
        'best_val_acc': best_val_acc,
        'best_epoch': best_epoch,
        'model_path': f'{output_dir}/psth_cluster_classifier_run_{run_idx+1}_best.pth'
    })
    print(f"  本次训练完成，最佳验证准确率: {best_val_acc:.4f} (epoch {best_epoch})\n")

print("=" * 60)
print("所有训练完成！")
print("=" * 60)

results_df = pd.DataFrame(all_runs_results)
results_df = results_df.sort_values('best_val_acc', ascending=False)
top_results = results_df.head(top_k)

print(f"\n验证准确率排名 (前{top_k}次):")
print(top_results.to_string(index=False))

# csv_path = f'{output_dir}/top_{top_k}_training_results.csv'
# top_results.to_csv(csv_path, index=False)
# print(f"\n前{top_k}次训练结果已保存到: {csv_path}")



开始重复训练，共 15 次，每次 10 个epoch

训练第 1/15 次
  本次训练完成，最佳验证准确率: 0.3300 (epoch 9)

训练第 2/15 次
  本次训练完成，最佳验证准确率: 0.3000 (epoch 3)

训练第 3/15 次
  本次训练完成，最佳验证准确率: 0.2700 (epoch 6)

训练第 4/15 次
  本次训练完成，最佳验证准确率: 0.2700 (epoch 4)

训练第 5/15 次
  本次训练完成，最佳验证准确率: 0.3200 (epoch 8)

训练第 6/15 次
  本次训练完成，最佳验证准确率: 0.3200 (epoch 2)

训练第 7/15 次
  本次训练完成，最佳验证准确率: 0.2900 (epoch 8)

训练第 8/15 次
  本次训练完成，最佳验证准确率: 0.2700 (epoch 4)

训练第 9/15 次
  本次训练完成，最佳验证准确率: 0.2600 (epoch 5)

训练第 10/15 次
  本次训练完成，最佳验证准确率: 0.2700 (epoch 10)

训练第 11/15 次
  本次训练完成，最佳验证准确率: 0.2900 (epoch 10)

训练第 12/15 次
  本次训练完成，最佳验证准确率: 0.3200 (epoch 8)

训练第 13/15 次
  本次训练完成，最佳验证准确率: 0.3100 (epoch 5)

训练第 14/15 次
  本次训练完成，最佳验证准确率: 0.2700 (epoch 5)

训练第 15/15 次
  本次训练完成，最佳验证准确率: 0.2600 (epoch 10)

所有训练完成！

验证准确率排名 (前5次):
 run_index  best_val_acc  best_epoch                                                                                                              model_path
         1          0.33           9  /home/ubuntu/Documents/jct/project/sort

In [46]:
# 为所有 stimulus_unique 中的图片生成类别标签 (0-6)
all_image_labels = np.zeros(len(stimulus_unique), dtype=int)
class_to_images = {i: [] for i in range(7)}

for stim_idx, img_name in enumerate(stimulus_unique):
    if img_name in image_path_dict:
        path = image_path_dict[img_name]
        class_name = os.path.basename(os.path.dirname(path))
        if class_name in class_to_cluster:
            class_id = class_to_cluster[class_name]
            all_image_labels[stim_idx] = class_id
            class_to_images[class_id].append(img_name)
        else:
            all_image_labels[stim_idx] = -1  # 未知类别
    else:
        all_image_labels[stim_idx] = -1  # 无路径

print(f"所有图片类别标签形状: {all_image_labels.shape}")
print(f"各类别数量: {np.bincount(all_image_labels[all_image_labels >= 0], minlength=7)}")
print(f"未知类别数量: {np.sum(all_image_labels == -1)}")

print("\n" + "=" * 60)
print("各类别的图片名称:")
print("=" * 60)
for class_id in range(7):
    images_in_class = class_to_images[class_id]
    print(f"\n类别 {class_id} ({len(images_in_class)} 张图片):")
    print("-" * 40)
    for img_name in images_in_class[:10]:
        print(f"  - {img_name}")
    if len(images_in_class) > 10:
        print(f"  ... 还有 {len(images_in_class) - 10} 张图片")

所有图片类别标签形状: (1000,)
各类别数量: [ 28 176  32  95 195 217 257]
未知类别数量: 0

各类别的图片名称:

类别 0 (28 张图片):
----------------------------------------
  - oven_oven_10s.jpg
  - lampshade_lampshade_05s.jpg
  - vent_vent_07s.jpg
  - railing_railing_02s.jpg
  - backdrop_backdrop_14s.jpg
  - air_conditioner_air_conditioner_13s.jpg
  - paper_plate_paper_plate_03s.jpg
  - blind_blind_08s.jpg
  - filter_filter_09s.jpg
  - cage_cage_12s.jpg
  ... 还有 18 张图片

类别 1 (176 张图片):
----------------------------------------
  - teddy_bear_teddy_bear_07s.jpg
  - seahorse_seahorse_06s.jpg
  - hamster_hamster_06s.jpg
  - bat1_bat1_02s.jpg
  - puppy_puppy_01b.jpg
  - hawk_hawk_04s.jpg
  - raccoon_raccoon_05s.jpg
  - fly_fly_03s.jpg
  - antelope_antelope_12n.jpg
  - eagle_eagle_07s.jpg
  ... 还有 166 张图片

类别 2 (32 张图片):
----------------------------------------
  - blouse_blouse_12s.jpg
  - outfit_outfit_11s.jpg
  - navel_navel_05s.jpg
  - sock_sock_11s.jpg
  - t-shirt_t-shirt_10s.jpg
  - thumb_thumb_01b.jpg
  - leotard_leotard

In [47]:
with open("/media/ubuntu/sda/Monkey/semantic/epoch_10_monkeyN/merged_cluster_class_counts.pkl", 'rb')as f:
    a = pickle.load(f)

In [22]:
# 计算每个 unit 的 class selectivity (SI)
# 每个类别随机抽取28个image参与计算，重复100次取平均

n_neurons = response1_to_baseline_ratio.shape[0]
n_classes = 7
n_samples_per_class = 28
n_iterations = 100

np.random.seed(42)

# 存储每次迭代的SI
all_si = np.zeros((n_iterations, n_neurons, n_classes), dtype=np.float32)

for iteration in range(n_iterations):
    si_iter = np.zeros((n_neurons, n_classes), dtype=np.float32)
    
    for neuron_idx in range(n_neurons):
        neuron_responses = response1_to_baseline_ratio[neuron_idx, :]  # (1000,)
        
        for class_id in range(n_classes):
            class_mask = all_image_labels == class_id
            class_indices = np.where(class_mask)[0]
            
            if len(class_indices) >= n_samples_per_class:
                sampled_indices = np.random.choice(class_indices, size=n_samples_per_class, replace=False)
                r_class = neuron_responses[sampled_indices]
            else:
                r_class = neuron_responses[class_mask]
            
            nonclass_mask = all_image_labels != class_id
            r_nonclass = neuron_responses[nonclass_mask]
            
            if len(r_class) > 0 and len(r_nonclass) > 0:
                mean_class = np.mean(r_class)
                mean_nonclass = np.mean(r_nonclass)
                var_class = np.var(r_class)
                var_nonclass = np.var(r_nonclass)
                
                denominator = np.sqrt(0.5 * (var_class + var_nonclass))
                if denominator > 0:
                    si = (mean_class - mean_nonclass) / denominator
                else:
                    si = 0.0
            else:
                si = 0.0
            
            si_iter[neuron_idx, class_id] = si
    
    all_si[iteration] = si_iter

# 计算100次的平均值
selectivity = np.mean(all_si, axis=0)

# print(f"Class selectivity 形状: {selectivity.shape}")  # (131, 7)
# print(f"基于 {n_iterations} 次随机采样（每次每类 {n_samples_per_class} 张）计算的平均值")

# print(f"\n各类别选择性统计:")
# for class_id in range(n_classes):
#     class_si = selectivity[:, class_id]
#     print(f"  Class {class_id}: mean={np.mean(class_si):.4f}, std={np.std(class_si):.4f}, "
#           f"max={np.max(class_si):.4f}, min={np.min(class_si):.4f}")

# # 保存 selectivity 结果
# selectivity_dict = {
#     'selectivity': selectivity,
#     'all_si': all_si,
#     'n_neurons': n_neurons,
#     'n_classes': n_classes,
#     'n_samples_per_class': n_samples_per_class,
#     'n_iterations': n_iterations,
#     'neuron_ids': filtered_neuron_ids
# }
# selectivity_path = f"{output_dir}/class_selectivity.pkl"
# with open(selectivity_path, 'wb') as f:
#     pickle.dump(selectivity_dict, f)
# print(f"\nClass selectivity 结果已保存: {selectivity_path}")


In [40]:
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from PIL import Image
import matplotlib.cm as cm

trials_per_page = 10
n_trials_to_plot = 100
n_time_bins_ms = 600

output_pdf_path = "/media/ubuntu/sda/duan/script/spike_sorting/raster_plot_trials.pdf"

trial_stim_points = rec_params['rec_codes_points'].astype(int).values
trial_start_times_ms = trial_stim_points[:n_trials_to_plot] * 0.1

responsive_neuron_indices = [all_neuron_ids.index(neuron_id) for neuron_id in filtered_neuron_ids]
n_responsive_neurons = len(filtered_neuron_ids)

filtered_raster_matrix = all_trial_raster_matrix[:, responsive_neuron_indices, :]

orange_to_blue_cmap = LinearSegmentedColormap.from_list('orange_to_blue', 
                                                         ['#fa973b', '#c9536b', '#8d3191', '#16265e'])

with PdfPages(output_pdf_path) as pdf:
    n_pages = (n_trials_to_plot + trials_per_page - 1) // trials_per_page
    
    for page_idx in range(n_pages):
        start_trial = page_idx * trials_per_page
        end_trial = min(start_trial + trials_per_page, n_trials_to_plot)
        
        page_start_time = trial_start_times_ms[start_trial]
        page_end_time = trial_start_times_ms[end_trial - 1] + n_time_bins_ms
        page_duration = page_end_time - page_start_time
        
        fig = plt.figure(figsize=(16, 6))
        
        gs = fig.add_gridspec(1, 2, width_ratios=[1, 15], wspace=0.02)
        ax_img = fig.add_subplot(gs[0])
        ax_raster = fig.add_subplot(gs[1])
        
        ax_img.set_xlim(0, 1)
        ax_img.set_ylim(0, n_responsive_neurons)
        ax_img.axis('off')
        ax_img.set_title('Trial Image', fontsize=10, pad=5)
        
        for trial_offset, trial_idx in enumerate(range(start_trial, end_trial)):
            trial_start_abs = trial_start_times_ms[trial_idx]
            
            img_name = stimulus_ids[trial_idx]
            img_path = image_path_dict.get(img_name, None)
            
            trial_raster = filtered_raster_matrix[trial_idx, :, :]
            
            for local_neuron_idx in range(n_responsive_neurons):
                spike_times = np.where(trial_raster[local_neuron_idx] > 0)[0]
                spike_times_abs = spike_times + trial_start_abs
                y_pos = local_neuron_idx
                
                color_ratio = 1 - (local_neuron_idx / max(1, n_responsive_neurons - 1))
                line_color = orange_to_blue_cmap(color_ratio)
                
                for spike_time in spike_times_abs:
                    ax_raster.plot([spike_time, spike_time], 
                                   [y_pos, y_pos + 0.8], 
                                   color=line_color, linewidth=2)
        
        for trial_idx in range(start_trial, end_trial):
            trial_start_abs = trial_start_times_ms[trial_idx]
            ax_raster.axvline(x=trial_start_abs, color='#696969', linestyle='--', 
                             linewidth=1.5, alpha=0.8)
        
        ax_raster.set_xlim(page_start_time - 50, page_end_time + 50)
        ax_raster.set_ylim(-0.5, n_responsive_neurons - 0.5)
        
        time_range = page_end_time - page_start_time
        if time_range > 1000:
            time_tick_step = 500
        elif time_range > 500:
            time_tick_step = 200
        else:
            time_tick_step = 100
        
        time_ticks = np.arange(page_start_time, page_end_time + 1, time_tick_step)
        ax_raster.set_xticks(time_ticks)
        ax_raster.set_xticklabels([str(int(t)) for t in time_ticks], fontsize=8)
        
        neuron_ticks = np.arange(0, n_responsive_neurons, 10)
        ax_raster.set_yticks(neuron_ticks)
        ax_raster.set_yticklabels([str(n) for n in neuron_ticks], fontsize=8)
        
        ax_raster.set_xlabel('Absolute Time (ms)', fontsize=12)
        ax_raster.set_ylabel('Neuron Index', fontsize=12)
        ax_raster.set_title(f'Trials {start_trial + 1} to {end_trial} - Raster Plot ({n_responsive_neurons} responsive neurons)', fontsize=14)
        
        sm = plt.cm.ScalarMappable(cmap=orange_to_blue_cmap, norm=plt.Normalize(vmin=0, vmax=n_responsive_neurons - 1))
        sm.set_array([])
        cbar = plt.colorbar(sm, ax=ax_raster, orientation='vertical', fraction=0.02, pad=0.02)
        cbar.set_label('Neuron Index', fontsize=10)
        
        plt.tight_layout()
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)
        
        print(f"已保存第 {page_idx + 1}/{n_pages} 页 (trials {start_trial + 1}-{end_trial})")

print(f"\nRaster plot 已保存到: {output_pdf_path}")
print(f"共 {n_pages} 页，每页 {trials_per_page} 个 trials")
print(f"使用 {n_responsive_neurons} 个有反应的神经元")
print(f"颜色渐变: 橙色(顶部 #fa973b) -> 粉紫色(#c9536b) -> 紫色(#8d3191) -> 深蓝色(底部 #16265e)")

已保存第 1/10 页 (trials 1-10)
已保存第 2/10 页 (trials 11-20)
已保存第 3/10 页 (trials 21-30)
已保存第 4/10 页 (trials 31-40)
已保存第 5/10 页 (trials 41-50)
已保存第 6/10 页 (trials 51-60)
已保存第 7/10 页 (trials 61-70)
已保存第 8/10 页 (trials 71-80)
已保存第 9/10 页 (trials 81-90)
已保存第 10/10 页 (trials 91-100)

Raster plot 已保存到: /media/ubuntu/sda/duan/script/spike_sorting/raster_plot_trials.pdf
共 10 页，每页 10 个 trials
使用 131 个有反应的神经元
颜色渐变: 橙色(顶部 #fa973b) -> 粉紫色(#c9536b) -> 紫色(#8d3191) -> 深蓝色(底部 #16265e)


In [ ]:
trials_per_page = 10
n_trials_to_plot = 100

output_pdf_path = "/media/ubuntu/sda/duan/script/spike_sorting/stimulus_images.pdf"

with PdfPages(output_pdf_path) as pdf:
    n_pages = (n_trials_to_plot + trials_per_page - 1) // trials_per_page
    
    for page_idx in range(n_pages):
        start_trial = page_idx * trials_per_page
        end_trial = min(start_trial + trials_per_page, n_trials_to_plot)
        n_current_trials = end_trial - start_trial
        
        fig, axes = plt.subplots(2, 5, figsize=(20, 8))
        axes = axes.flatten()
        
        for i, trial_idx in enumerate[int](range(start_trial, end_trial)):
            ax = axes[i]
            ax.axis('off')
            
            img_name = stimulus_ids[trial_idx]
            img_path = image_path_dict.get(img_name, None)
            
            ax.set_title(f"Trial {trial_idx + 1}\n{img_name}", fontsize=10)
            
            if img_path:
                try:
                    img = Image.open(img_path)
                    img.thumbnail((300, 300))
                    ax.imshow(img)
                except Exception as e:
                    ax.text(0.5, 0.5, f"Error loading:\n{img_name}", 
                           ha='center', va='center', fontsize=8)
                    ax.set_facecolor('#ffcccc')
            else:
                ax.text(0.5, 0.5, f"Missing:\n{img_name}", 
                       ha='center', va='center', fontsize=8)
                ax.set_facecolor('#ffcccc')
        
        for i in range(n_current_trials, len(axes)):
            axes[i].axis('off')
        
        plt.suptitle(f"Stimulus Images - Trials {start_trial + 1} to {end_trial}", 
                     fontsize=14, fontweight='bold')
        plt.tight_layout()
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)
        
        print(f"已保存第 {page_idx + 1}/{n_pages} 页 (trials {start_trial + 1}-{end_trial})")

print(f"\n刺激图片已保存到: {output_pdf_path}")
print(f"共 {n_pages} 页，每页 {trials_per_page} 张图片")

已保存第 1/10 页 (trials 1-10)
已保存第 2/10 页 (trials 11-20)
已保存第 3/10 页 (trials 21-30)
已保存第 4/10 页 (trials 31-40)
已保存第 5/10 页 (trials 41-50)
已保存第 6/10 页 (trials 51-60)
已保存第 7/10 页 (trials 61-70)
已保存第 8/10 页 (trials 71-80)
已保存第 9/10 页 (trials 81-90)
已保存第 10/10 页 (trials 91-100)

刺激图片已保存到: /media/ubuntu/sda/duan/script/spike_sorting/stimulus_images.pdf
共 10 页，每页 10 张图片


In [52]:
from PIL import Image

print("=" * 60)
print("为每个类别选取10张图片，生成类别示例PDF")
print("=" * 60)

output_pdf_path = "/media/ubuntu/sda/duan/script/spike_sorting/class_examples.pdf"

fig, axes = plt.subplots(7, 10, figsize=(20, 14))
axes = axes.flatten()

sample_idx = 0
for class_id in range(7):
    images_in_class = class_to_images[class_id]
    selected_images = images_in_class[10:20]
    
    for i, img_name in enumerate(selected_images):
        ax = axes[sample_idx]
        ax.axis('off')
        
        if img_name in image_path_dict:
            img_path = image_path_dict[img_name]
            try:
                img = Image.open(img_path)
                img.thumbnail((200, 200))
                ax.imshow(img)
            except Exception as e:
                ax.text(0.5, 0.5, "Error", ha='center', va='center')
                ax.set_facecolor('#ffcccc')
        else:
            ax.text(0.5, 0.5, "Missing", ha='center', va='center')
            ax.set_facecolor('#ffcccc')
        
        if i == 0:
            ax.set_ylabel(f'Class {class_id}', fontsize=12, fontweight='bold')
        
        sample_idx += 1

for i in range(sample_idx, len(axes)):
    axes[i].axis('off')

plt.suptitle('Example Images per Class (10 per class)', fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0.05, 0, 1, 0.95])

with PdfPages(output_pdf_path) as pdf:
    pdf.savefig(fig, bbox_inches='tight')

plt.close(fig)

print(f"\n类别示例图片已保存到: {output_pdf_path}")
print(f"共 7 个类别，每类 10 张图片")

为每个类别选取10张图片，生成类别示例PDF

类别示例图片已保存到: /media/ubuntu/sda/duan/script/spike_sorting/class_examples.pdf
共 7 个类别，每类 10 张图片


In [80]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

csv_path = "/home/ubuntu/Documents/jct/project/sorted/260121/groups_01_to_05_sliced/history/classification_accuracy.csv"
df = pd.read_csv(csv_path)
means = df.mean()
stds = df.std()

with PdfPages("/media/ubuntu/sda/duan/script/spike_sorting/classification_accuracy_comparison.pdf") as pdf:
    fig, ax = plt.subplots(figsize=(6, 4))

    x = np.arange(len(df.columns))
    bar_colors = ['#df6c7a', '#87ac7b', '#5e9fd1']
    bars = ax.bar(x, means.values, yerr=stds.values, capsize=8, 
                color=bar_colors, width = 0.3)

    ax.set_xlabel('Method', fontsize=12)
    ax.set_ylabel('Accuracy', fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(means.index, fontsize=11)
    ax.set_ylim(0, 0.4)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    pdf.savefig()
    plt.close()

    csv_path = "/home/ubuntu/Documents/jct/project/sorted/260121/groups_01_to_05_sliced/history/timing_results.csv"
    df = pd.read_csv(csv_path, index_col=0)
    means = df.mean()
    stds = df.std()

    fig, ax = plt.subplots(figsize=(6, 4))

    x = np.arange(len(df.columns))
    bar_colors = ['#df6c7a']
    bars = ax.bar(x, means.values, yerr=stds.values, capsize=8, 
                color=bar_colors, width = 0.1)

    ax.set_xticks(x)
    ax.set_xticklabels(means.index, fontsize=11)
    ax.set_ylim(0, 0.4)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    pdf.savefig()
    plt.close()

In [74]:
df

,duration_seconds
trial_idx,
4000,0.495719
4001,0.194650
4002,0.193549
4003,0.195923
4004,0.188164
...,...
4995,0.196197
4996,0.191768
4997,0.190913


In [44]:
print("=" * 60)
print("使用筛选前的PSTH矩阵(5000 trials, 131 neurons)进行训练")
print("=" * 60)

n_trials_to_use = 5000
n_neurons_to_use = len(filtered_neuron_ids)

print(f"\n使用数据形状: ({n_trials_to_use}, {n_neurons_to_use}, 30)")

responsive_neuron_indices = [all_neuron_ids.index(neuron_id) for neuron_id in filtered_neuron_ids]

psth_data = all_trial_psth_matrix[:n_trials_to_use, responsive_neuron_indices, :]

print(f"PSTH数据形状: {psth_data.shape}")

stimulus_ids_5000 = stimulus_ids[:n_trials_to_use]

image_to_class = {}
for stim_idx, img_name in enumerate(stimulus_unique):
    if img_name in image_path_dict:
        path = image_path_dict[img_name]
        class_name = os.path.basename(os.path.dirname(path))
        if class_name in class_to_cluster:
            image_to_class[img_name] = class_to_cluster[class_name]

trial_labels = np.array([image_to_class.get(img_name, -1) for img_name in stimulus_ids_5000])

valid_mask = trial_labels >= 0
psth_valid = psth_data[valid_mask]
labels_valid = trial_labels[valid_mask]

print(f"有效样本数量: {len(labels_valid)}")
print(f"有效PSTH形状: {psth_valid.shape}")
print(f"各类数量: {np.bincount(labels_valid, minlength=7)}")

train_indices, val_indices = train_test_split(
    range(len(psth_valid)),
    test_size=0.1,
    random_state=42,
    stratify=labels_valid
)

train_psth = psth_valid[train_indices]
train_labels = labels_valid[train_indices]
val_psth = psth_valid[val_indices]
val_labels = labels_valid[val_indices]

print(f"\n训练集: {len(train_psth)} samples")
print(f"验证集: {len(val_psth)} samples")

train_dataset = PSTHClusterDataset(train_psth, train_labels)
val_dataset = PSTHClusterDataset(val_psth, val_labels)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)

n_neurons_model = train_psth.shape[1]
n_time_bins_model = train_psth.shape[2]

num_runs = 15
num_epochs = 10
top_k = 5
all_runs_results = []

print(f"\n开始重复训练，共 {num_runs} 次，每次 {num_epochs} 个epoch\n")

for run_idx in range(num_runs):
    print(f"=" * 60)
    print(f"训练第 {run_idx + 1}/{num_runs} 次")
    print(f"=" * 60)
    
    torch.manual_seed(42 + run_idx)
    np.random.seed(42 + run_idx)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(42 + run_idx)
    
    encoder_backbone = TemporalEPEncoder(
        input_dim=n_neurons_model,
        time_bins=n_time_bins_model,
        d_model=64,
        n_token=128,
        num_conv_layers=2,
        dropout=0.2,
        output_dim=768
    )
    model = PSTHClusterClassifier(encoder_backbone, num_classes=7).to(device)
    ce_criterion = nn.CrossEntropyLoss()
    infonce_criterion = InfoNCELoss(temperature=0.07, num_classes=7)
    alpha = 0.5
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    
    best_val_acc = 0.0
    best_epoch = 0
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for psth_batch, label_batch in train_loader:
            psth_batch = psth_batch.to(device)
            label_batch = label_batch.to(device)

            optimizer.zero_grad()
            logits, features = model(psth_batch, return_features=True)
            ce_loss = ce_criterion(logits, label_batch)
            infonce_loss = infonce_criterion(features, label_batch)
            loss = alpha * ce_loss + (1 - alpha) * infonce_loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item() * psth_batch.size(0)
            pred = logits.argmax(dim=1)
            train_correct += (pred == label_batch).sum().item()
            train_total += label_batch.size(0)

        avg_train_loss = train_loss / train_total
        train_acc = train_correct / train_total
        scheduler.step()

        model.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for psth_batch, label_batch in val_loader:
                psth_batch = psth_batch.to(device)
                label_batch = label_batch.to(device)
                logits = model(psth_batch)
                pred = logits.argmax(dim=1)
                val_correct += (pred == label_batch).sum().item()
                val_total += label_batch.size(0)

        val_acc = val_correct / val_total

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch + 1
            torch.save(model.state_dict(), f'{output_dir}/single_trial_classifier_run_{run_idx+1}_best.pth')
    
    all_runs_results.append({
        'run_index': run_idx + 1,
        'best_val_acc': best_val_acc,
        'best_epoch': best_epoch,
        'model_path': f'{output_dir}/single_trial_classifier_run_{run_idx+1}_best.pth'
    })
    print(f"  本次训练完成，最佳验证准确率: {best_val_acc:.4f} (epoch {best_epoch})\n")

print("=" * 60)
print("所有训练完成！")
print("=" * 60)

results_df = pd.DataFrame(all_runs_results)
results_df = results_df.sort_values('best_val_acc', ascending=False)
top_results = results_df.head(top_k)

print(f"\n验证准确率排名 (前{top_k}次):")
print(top_results.to_string(index=False))

csv_path = f'{output_dir}/single_trial_top_{top_k}_results.csv'
top_results.to_csv(csv_path, index=False)
print(f"\n前{top_k}次训练结果已保存到: {csv_path}")

使用筛选前的PSTH矩阵(5000 trials, 131 neurons)进行训练

使用数据形状: (5000, 131, 30)
PSTH数据形状: (5000, 131, 600)
有效样本数量: 5000
有效PSTH形状: (5000, 131, 600)
各类数量: [ 142  879  160  473  976 1084 1286]

训练集: 4500 samples
验证集: 500 samples

开始重复训练，共 15 次，每次 10 个epoch

训练第 1/15 次
  本次训练完成，最佳验证准确率: 0.2860 (epoch 4)

训练第 2/15 次
  本次训练完成，最佳验证准确率: 0.2660 (epoch 7)

训练第 3/15 次
  本次训练完成，最佳验证准确率: 0.2860 (epoch 7)

训练第 4/15 次
  本次训练完成，最佳验证准确率: 0.2940 (epoch 4)

训练第 5/15 次
  本次训练完成，最佳验证准确率: 0.3080 (epoch 7)

训练第 6/15 次
  本次训练完成，最佳验证准确率: 0.2900 (epoch 10)

训练第 7/15 次
  本次训练完成，最佳验证准确率: 0.3040 (epoch 10)

训练第 8/15 次
  本次训练完成，最佳验证准确率: 0.2660 (epoch 1)

训练第 9/15 次
  本次训练完成，最佳验证准确率: 0.2960 (epoch 5)

训练第 10/15 次
  本次训练完成，最佳验证准确率: 0.2940 (epoch 6)

训练第 11/15 次
  本次训练完成，最佳验证准确率: 0.2640 (epoch 9)

训练第 12/15 次
  本次训练完成，最佳验证准确率: 0.2900 (epoch 6)

训练第 13/15 次
  本次训练完成，最佳验证准确率: 0.2880 (epoch 8)

训练第 14/15 次
  本次训练完成，最佳验证准确率: 0.2900 (epoch 6)

训练第 15/15 次
  本次训练完成，最佳验证准确率: 0.2840 (epoch 7)

所有训练完成！

验证准确率排名 (前5次):
 run_index  best_val